# Week3_ex5 - BLI Force Simulation

Put a straight wire inside a uniform external field (built with tangential H-field boundary conditions on a bounding box) and used assign_force with the Lorentz method to check the force on the wire matches the expected B*I*L. Magnetostatic solve, so none of the EddyCurrent/AC Magnetic workarounds from ex2-ex4 apply here.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil


In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create project and set solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style API calls
oDesign = M3D.odesign


In [ ]:
## Set up output directory for results ##

proj_name = "Week3_ex5"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week3_ex5"

## Save project and apply design name ##

proj = M3D.oproject

proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()


In [ ]:
# Helper to find the two terminal faces of a wire/coil object
# Looks at face center x-coordinates and picks the two faces with the largest |x| (the wire ends)
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position among all faces
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # collect the faces sitting at that max x position
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort by x so we consistently get (positive-x face, negative-x face)
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
line_length = 500
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"

B_field = 1
mu_0 = 4*math.pi*1e-7
H_field = B_field/mu_0
M3D["H_field"] = f"{H_field}"


In [ ]:
## Create and configure analysis setup ##

my_setup = M3D.create_setup(name="Setup1")

my_setup.props["MaximumPasses"] = 20
my_setup.props["PercentRefinement"] = 30
my_setup.props["PercentError"] = 1
my_setup.props["RelativeResidual"] = 1e-06
my_setup.props["NonLinearResidual"] = 0.001
my_setup.props["SmoothBHCurve"] = False
# NOTE: this nested BH-curve option was set via a raw "MuOption" block in the original
# script-recorded setup ("MuOption" -> "MuNonLinearBH": True). Best-guess translation to
# pyaedt's props dict below -- verify locally that this actually takes effect (check
# my_setup.props after update(), or compare solve behavior against the original raw version).
my_setup.props["MuOption"] = {"MuNonLinearBH": True}
my_setup.update()


In [ ]:
# 1. draw the straight wire, split into a short middle section (line_cal) where we
# actually measure the force -- keeps the force calc close to an ideal uniform-field wire
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

point_tmp = []
point_tmp.append(["-50mm", "0mm", "0mm"])
point_tmp.append(["50mm", "0mm", "0mm"])
line_cal = M3D.modeler.create_polyline(points=point_tmp, name="line_cal", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

M3D.modeler.subtract(blank_list=[line], tool_list=[line_cal], keep_originals=True)


# 2. create the box where the uniform field is enforced
origin = ["-line_length/2+10mm", "-line_length/2", "-line_length/2"]
sizes = ["line_length-20mm", "line_length", "line_length"]
box_h = M3D.modeler.create_box(origin=origin, sizes=sizes, name="box_h", material=None)
M3D.modeler.subtract(blank_list=[box_h], tool_list=[line, line_cal], keep_originals=True)


# 3. create the outer region
pad_value = ["line_length/2", "-line_length/2", "line_length/2+5mm", "-line_length/2-5mm", "line_length/2+5mm", "-line_length/2-5mm"]
region = M3D.modeler.create_region(pad_value=pad_value, pad_type='Absolute Position', name='Region')


In [ ]:
# apply a tangential H-field boundary (y-direction) on the faces parallel to the y-axis
tmp = [box_h.top_face_x, box_h.bottom_face_x, box_h.top_face_z, box_h.bottom_face_z]    
for f in tmp : 
    origin = f.center
    u_pos = origin.copy()
    u_pos[1] = u_pos[1] - 10
    M3D.assign_tangential_h_field(assignment=f, x_component_real=H_field, coordinate_system='Global', origin=origin, u_pos=u_pos)


# zero out the tangential H-field on the faces perpendicular to y, so only the
# normal component survives there (keeps the field uniform inside box_h)
# NOTE: M3D's wrapper for this had a bug, so this uses the raw script-recording call instead
oModule = oDesign.GetModule("BoundarySetup")
oModule.AssignZeroTangentialHField(
	[
		"NAME:ZeroTangentialHField1",
		"Faces:="		, [box_h.top_face_y.id, box_h.bottom_face_y.id]
	])


In [ ]:
# use Lorentz force here rather than virtual force -- accurate with a much coarser mesh
# since the wire's relative permeability is essentially 1
force_line = M3D.assign_force(assignment=line_cal, coordinate_system='Global', is_virtual=False, force_name=None)


In [ ]:
## mesh setup ##

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:line_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["line_cal"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_diameter/4"
	])


In [ ]:
# 1. set the wire's two end faces as coil terminals
ter_out, ter_in = find_terminal_face(line)

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Positive", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Negative", name="In")

# 2. create a winding and add the coils to it
coil = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                   resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                   name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])


In [ ]:
my_setup.analyze()


In [ ]:
oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Force Table 1", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [], 
	[
		"line_length:="		, ["All"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"],
		"H_field:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "line_length",
		"Y Component:="		, [f"{force_line.name}.Force_mag",f"{force_line.name}.Force_x",f"{force_line.name}.Force_y",f"{force_line.name}.Force_z"]
	])


In [ ]:
## save project ##

M3D.save_project()
